<a href="https://colab.research.google.com/github/hetaf234/GP_Layers/blob/main/Layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Layers

## Loading And Joining

### Loading From Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd
import os

BASE_PATH = '/content/drive/MyDrive/Graduation_Project/Layers'
LAYER1_PATH = os.path.join(BASE_PATH, 'layer1.json')
LAYER2_PATH = os.path.join(BASE_PATH, 'layer2.json')

print(os.path.exists(LAYER1_PATH), os.path.exists(LAYER2_PATH))

Mounted at /content/drive
True True


In [2]:
with open(LAYER1_PATH, 'r') as f:
    layer1 = json.load(f)

print(f"Total recipes in layer1: {len(layer1)}")
print(layer1[0])  # inspect one entry to confirm structure

Total recipes in layer1: 1029720
{'ingredients': [{'text': '6 ounces penne'}, {'text': '2 cups Beechers Flagship Cheese Sauce (recipe follows)'}, {'text': '1 ounce Cheddar, grated (1/4 cup)'}, {'text': '1 ounce Gruyere cheese, grated (1/4 cup)'}, {'text': '1/4 to 1/2 teaspoon chipotle chili powder (see Note)'}, {'text': '1/4 cup (1/2 stick) unsalted butter'}, {'text': '1/3 cup all-purpose flour'}, {'text': '3 cups milk'}, {'text': '14 ounces semihard cheese (page 23), grated (about 3 1/2 cups)'}, {'text': '2 ounces semisoft cheese (page 23), grated (1/2 cup)'}, {'text': '1/2 teaspoon kosher salt'}, {'text': '1/4 to 1/2 teaspoon chipotle chili powder'}, {'text': '1/8 teaspoon garlic powder'}, {'text': '(makes about 4 cups)'}], 'url': 'http://www.epicurious.com/recipes/food/views/-world-s-best-mac-and-cheese-387747', 'partition': 'train', 'title': 'Worlds Best Mac and Cheese', 'id': '000018c8a5', 'instructions': [{'text': 'Preheat the oven to 350 F. Butter or oil an 8-inch baking dish.'}

In [3]:
with open(LAYER2_PATH, 'r') as f:
    layer2 = json.load(f)

print(f"Total entries in layer2: {len(layer2)}")
print(layer2[0])  # inspect one entry

Total entries in layer2: 402760
{'id': '00003a70b1', 'images': [{'id': '3e233001e2.jpg', 'url': 'http://img.sndimg.com/food/image/upload/w_512,h_512,c_fit,fl_progressive,q_95/v1/img/recipes/47/91/49/picaYYmb9.jpg'}, {'id': '7f749987f9.jpg', 'url': 'http://img.sndimg.com/food/image/upload/w_512,h_512,c_fit,fl_progressive,q_95/v1/img/recipes/47/91/49/picpy37SW.jpg'}, {'id': 'aaf6b2dcd3.jpg', 'url': 'http://img.sndimg.com/food/image/upload/w_512,h_512,c_fit,fl_progressive,q_95/v1/img/recipes/47/91/49/picX9CNE2.jpg'}]}


### Cleaning - step 1
here we dropped rows that has empty (Title ,Ingrediets , and has no image)

In [4]:
# Build id -> list of image ids from layer2
image_lookup = {}
for entry in layer2:
    recipe_id = entry['id']
    images = entry.get('images', [])
    if images:
        image_lookup[recipe_id] = [img['id'] for img in images]

print(f"Recipes with at least one image: {len(image_lookup)}")

# Join with layer1
rows = []
for recipe in layer1:
    rid = recipe['id']
    title = recipe.get('title', '').strip()
    ingredients = [ing['text'] for ing in recipe.get('ingredients', [])]
    partition = recipe.get('partition', '')
    images = image_lookup.get(rid, [])

    if title and ingredients and images:
        rows.append({
            'id': rid,
            'title': title,
            'ingredients': ingredients,
            'num_ingredients': len(ingredients),
            'image_ids': images,
            'partition': partition
        })

df = pd.DataFrame(rows)
print(f"Final clean dataset size: {len(df)}")
df.head()

Recipes with at least one image: 402760
Final clean dataset size: 402760


,id,title,ingredients,num_ingredients,image_ids,partition
0,00003a70b1,Crunchy Onion Potato Bake,"[2 12 cups milk, 1 12 cups water, 14 cup butte...",7,"[3e233001e2.jpg, 7f749987f9.jpg, aaf6b2dcd3.jpg]",test
1,000075604a,Kombu Tea Grilled Chicken Thigh,"[2 Chicken thighs, 2 tsp Kombu tea, 1 White pe...",3,[6bdca6e490.jpg],train
2,00007bfd16,Strawberry Rhubarb Dump Cake,"[6 -8 cups fresh rhubarb, or, 6 -8 cups frozen...",7,"[6409eab844.jpg, f7cb3de295.jpg]",train
3,000095fc1d,Yogurt Parfaits,"[8 ounces, weight Light Fat Free Vanilla Yogur...",3,[a1374cdd98.jpg],train
4,0000b1e2b5,Fennel-Rubbed Pork Tenderloin with Roasted Fen...,"[1 teaspoon fennel seeds, 1 pound pork tenderl...",9,[cb1a684683.jpg],train


In [5]:

#count partitions
print(df['partition'].value_counts())
print(df['title'].sample(10).tolist())
print(df['num_ingredients'].describe())

partition
train    281598
test      60740
val       60422
Name: count, dtype: int64
['Firecracker Burgers With Cooling Lime Sauce', 'Billy Sunday Summer Cobbler Cocktail', 'Un-Chicken Noodle Soup (Raw Foods)', 'Banana-Sour Cream Cake With Cream Cheese Frosting', 'Cabbage and Sausage Crock Pot', 'Sausage Gravy', 'Beef and Bacon hot-pot', 'Whats Up, G!', 'Slow Cooker French Onion Beef Soup', 'Grape-tini']
count    402760.000000
mean          9.205564
std           4.146942
min           1.000000
25%           6.000000
50%           9.000000
75%          12.000000
max          99.000000
Name: num_ingredients, dtype: float64


## Save the cleaned data

In [6]:
SAVE_PATH = os.path.join(BASE_PATH, '..', 'processed')
os.makedirs(SAVE_PATH, exist_ok=True)

df.to_pickle(os.path.join(SAVE_PATH, 'dataset.pkl'))
print("Saved to:", os.path.join(SAVE_PATH, 'dataset.pkl'))

Saved to: /content/drive/MyDrive/Graduation_Project/Layers/../processed/dataset.pkl


## Check Data Quality

In [7]:
# Duplicate titles
print("Unique titles:", df['title'].nunique())
print("Total rows:", len(df))
print("Most common titles:")
print(df['title'].value_counts().head(20))

Unique titles: 341958
Total rows: 402760
Most common titles:
title
Banana Bread              91
Guacamole                 88
Chocolate Chip Cookies    70
Chicken Enchiladas        67
Chicken Tortilla Soup     62
Apple Crisp               62
Broccoli Salad            60
French Onion Soup         58
Peanut Butter Cookies     56
Chicken Pot Pie           50
Cucumber Salad            47
Taco Soup                 47
Zucchini Bread            46
Sloppy Joes               46
Meatloaf                  46
Chicken Cacciatore        45
Deviled Eggs              42
Snickerdoodles            41
Carrot Cake               41
Key Lime Pie              40
Name: count, dtype: int64


In [8]:
# Ingredient count distribution
print(df['num_ingredients'].describe())
print(df[df['num_ingredients'] <= 2].shape[0], "recipes with <=2 ingredients")
print(df[df['num_ingredients'] >= 30].shape[0], "recipes with >=30 ingredients")

count    402760.000000
mean          9.205564
std           4.146942
min           1.000000
25%           6.000000
50%           9.000000
75%          12.000000
max          99.000000
Name: num_ingredients, dtype: float64
5017 recipes with <=2 ingredients
346 recipes with >=30 ingredients


In [9]:
# Ingredient count distribution
print(df['num_ingredients'].describe())
print(df[df['num_ingredients'] <= 2].shape[0], "recipes with <=2 ingredients")
print(df[df['num_ingredients'] >= 30].shape[0], "recipes with >=30 ingredients")

count    402760.000000
mean          9.205564
std           4.146942
min           1.000000
25%           6.000000
50%           9.000000
75%          12.000000
max          99.000000
Name: num_ingredients, dtype: float64
5017 recipes with <=2 ingredients
346 recipes with >=30 ingredients


In [10]:
# Non-ASCII / non-English check (rough heuristic)
non_ascii = df[df['title'].apply(lambda t: not t.isascii())]
print(f"Titles with non-ASCII characters: {len(non_ascii)}")
print(non_ascii['title'].head(10).tolist())

Titles with non-ASCII characters: 0
[]
